In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CIFAR-10 dataset (images are 32x32)
transform = transforms.Compose([
    transforms.Resize(224),  # AlexNet expects 224x224 input
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, 
                                             transform=transform, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, 
                                            transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

In [5]:
alexnet = models.alexnet(pretrained=True)
print(alexnet)

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

# Task 2.1.2

In [6]:
# Load pretrained AlexNet
alexnet_pretrained = models.alexnet(pretrained=True)

# Freeze all layers
for param in alexnet_pretrained.parameters():
    param.requires_grad = False

# Add extra fully connected layer for CIFAR-10
alexnet_pretrained.classifier = nn.Sequential(
    *alexnet_pretrained.classifier,  # keep all original layers
    nn.Linear(1000, 10)              # new trainable layer
)

alexnet_pretrained = alexnet_pretrained.to(device)

# Only train the last layer (the new 1000→10 layer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(alexnet_pretrained.classifier[7].parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    alexnet_pretrained.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = alexnet_pretrained(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')

Epoch [1/10], Loss: 1.0615
Epoch [2/10], Loss: 0.9472
Epoch [3/10], Loss: 0.9177
Epoch [4/10], Loss: 0.9174
Epoch [5/10], Loss: 0.8997
Epoch [6/10], Loss: 0.8856
Epoch [7/10], Loss: 0.8924
Epoch [8/10], Loss: 0.8809
Epoch [9/10], Loss: 0.8778
Epoch [10/10], Loss: 0.8797


In [7]:
alexnet_pretrained.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = alexnet_pretrained(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Test Accuracy (Feature Extraction): {100 * correct / total:.2f}%')

Test Accuracy (Feature Extraction): 73.60%


Interpretation: 
i) Feature Extraction accuracy is slightly lower because only the new FC layer learns. 
ii) Pretrained convolutional layers are frozen, so the network relies purely on ImageNet features.